In [1]:
from sampo.schemas.graph import WorkGraph
from scripts.wg_converter import WorkGraphConverter
from sampo_api import contractor

def read_file(path='wgs/small_synth', file='wg_9', contractors_N=5):  
    wg = WorkGraph.loadf(path, file)
    contractors = contractor(N = contractors_N)
    conv = WorkGraphConverter()
    data = conv.convert(wg, contractors)['rcpsp_data']
    return data


wg = WorkGraph.loadf('wgs/small_synth', 'wg_9')
data = read_file()
from sampo.scheduler.lft import LFTScheduler
contractors = contractor(N=5)



sample_schedule_df = LFTScheduler().schedule(wg, contractors)[0].full_schedule_df[1:20]
wg_df = wg.to_frame(save_req=True)

Can not find native module; switching to default


In [2]:
import ast
from typing import Dict, Optional

def extract_compute_duration(path: str) -> Optional[str]:
    with open(path, "r", encoding="utf-8") as f:
        source = f.read()
    
    tree = ast.parse(source)
    lines = source.splitlines()

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == "compute_duration":
            # Дополнительно можно проверить сигнатуру
            arg_names = [a.arg for a in node.args.args]
            if arg_names[:3] != ["job", "contractor", "allocated"]:
                continue
            
            # В Python 3.8+ у узлов есть lineno и end_lineno
            start = node.lineno - 1          # lineno 1-based
            end = node.end_lineno            # end_lineno включительно
            return "\n".join(lines[start:end])
    
    return None


# Проверка для вычисления времени
print( extract_compute_duration('Heuristics/deepseek_chat/Earliest Finish Time Priority Rule (EFT)') )


    def compute_duration(job: Dict, contractor: Dict, allocated: Dict[str, int]) -> Optional[int]:
        """Compute job duration for given contractor and allocation."""
        duration = 0
        for r, volume in job['work_volume'].items():
            if volume == 0:
                continue
            if r not in contractor['workers']:
                return None
            if allocated[r] == 0:
                # If no workers allocated but work_volume > 0, invalid
                return None
            
            if communication_coefficient:
                productivity = allocated[r] * communication_coefficient(allocated[r], job['demand_max'][r])
            else:
                productivity = allocated[r]
            
            if productivity <= 0:
                return None
            
            resource_duration = math.ceil(volume / productivity)
            duration = max(duration, resource_duration)
        return duration if duration > 0 else None


In [3]:
import pandas as pd


df_total = pd.merge(sample_schedule_df, 
         wg_df[['activity_id', 'req_volume', 'max_req']], 
         left_on='task_id', right_on ='activity_id')


In [4]:
import math
import ast

def transform(dict_values):
     if not isinstance(dict_values, dict):
          return  ast.literal_eval(dict_values)
     else:
          return dict_values
def compute_duration(r) -> int:
        """Compute job duration for given contractor and allocation."""
        req_volume, allocated, max_req = r
        req_volume = transform(req_volume)
        allocated = transform(allocated)
        max_req = transform(max_req)
        # if isinstance(req_volume, str), req_volume)
        duration = 0
        for r, volume in req_volume.items(): # work_volume -> req_volume
            if volume == 0:
                continue
            # if r not in contractor['workers']:
            #     return None
            if allocated[r] == 0:
                # If no workers allocated but work_volume > 0, invalid
                return None
            
            if communication_coefficient:
                productivity = allocated[r] * communication_coefficient(allocated[r], max_req[r]) # demand_max -> max_req
            else:
                productivity = allocated[r]
            
            if productivity <= 0:
                return None
            
            resource_duration = math.ceil(volume / productivity)
            duration = max(duration, resource_duration)
        return duration if duration > 0 else None

communication_coefficient = lambda n, m: 1.0 / (6 * m ** 2) * (-2 * n ** 3 + 3 * n ** 2 + (6 * m ** 2 - 1) * n) if m != 0 else 0.0

In [5]:
df_total['duration_in_llm'] = df_total[['req_volume', 'workers', 'max_req']].apply(compute_duration,axis=1)

In [6]:
(df_total['duration'] == df_total['duration_in_llm']).mean()

1.0